# MEaMt-Net: Semi-Supervised Liver Tumor Segmentation & Quantification

**Contrastive-Driven and Uncertainty-Guided Joint Learning on Non-Contrast MRI**

This notebook walks through:
1. Environment & Configuration
2. Data Loading & Visualization
3. Model Architecture
4. Loss Functions
5. Training Loop (mini demo)
6. Inference & Evaluation
7. Results Visualization

## 1. Environment & Configuration

In [ ]:
import os, sys
# Make sure project root is on the path
ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"})')
print(f'Project  : {ROOT}')

In [ ]:
from configs.default import load_config

cfg = load_config('configs/default.yaml')

print('=== Dataset ===')
for k, v in cfg['dataset'].items():
    print(f'  {k}: {v}')
print('=== Training ===')
for k, v in cfg['training'].items():
    print(f'  {k}: {v}')
print('=== Loss Weights ===')
for k, v in cfg['loss'].items():
    print(f'  {k}: {v}')

## 2. Data Loading & Visualization

In [ ]:
# ── Synthetic data generator (runs without the real dataset) ──────────────────
def make_synthetic_patient(data_dir, patient_id, H=256, W=256):
    """Create a fake patient folder with synthetic MRI slices and annotations."""
    pdir = Path(data_dir) / patient_id
    pdir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(abs(hash(patient_id)) % (2**31))

    # Liver-like background
    t2fs = rng.random((H, W)).astype(np.float32) * 0.4 + 0.1
    dwi  = rng.random((H, W)).astype(np.float32) * 0.3 + 0.05

    # Circular tumour
    cx, cy, r = rng.integers(80, 180), rng.integers(80, 180), rng.integers(15, 35)
    yy, xx = np.ogrid[:H, :W]
    mask = ((xx - cx)**2 + (yy - cy)**2 <= r**2).astype(np.int64)
    t2fs[mask == 1] += rng.random(mask.sum()).astype(np.float32) * 0.4 + 0.3
    dwi[mask == 1]  += rng.random(mask.sum()).astype(np.float32) * 0.5 + 0.4

    t2fs = t2fs.clip(0, 1)
    dwi  = dwi.clip(0, 1)

    # Quantification targets (normalized)
    quant = np.array([cx / W, cy / H, (np.pi * r**2) / (H * W)], dtype=np.float32)

    np.save(pdir / 'T2FS.npy',  t2fs)
    np.save(pdir / 'DWI.npy',   dwi)
    np.save(pdir / 'seg.npy',   mask)
    np.save(pdir / 'quant.npy', quant)
    return pdir


# Generate synthetic dataset
DATA_ROOT = Path('data/LLD-MMRI')
for split, n in [('train', 20), ('val', 5), ('test', 5)]:
    for i in range(n):
        make_synthetic_patient(DATA_ROOT / split, f'patient_{i+1:03d}')

print(f'Synthetic dataset created at {DATA_ROOT.resolve()}')

In [ ]:
from data.dataset import MedicalImageDataset
from data.transforms import get_transforms
from torch.utils.data import DataLoader

train_tfms, val_tfms = get_transforms((256, 256))

train_ds = MedicalImageDataset(DATA_ROOT, mode='train', transforms=train_tfms, label_ratio=0.2)
val_ds   = MedicalImageDataset(DATA_ROOT, mode='val',   transforms=val_tfms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=0)

print(f'Train samples : {len(train_ds)}  (labeled ≈ {int(len(train_ds)*0.2)})')
print(f'Val   samples : {len(val_ds)}')

In [ ]:
# Visualise one batch
batch = next(iter(train_loader))
imgs  = batch['image']   # [B, 2, H, W]
segs  = batch['seg']     # [B, H, W]
quant = batch['quant']   # [B, 3]
ids   = batch['id']

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
titles = ['T2FS', 'DWI', 'Segmentation mask', 'Overlay']
for col, title in enumerate(titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

cmap_seg = plt.cm.get_cmap('Reds', 2)

for row in range(4):
    t2  = imgs[row, 0].numpy()
    dwi = imgs[row, 1].numpy()
    seg = segs[row].numpy()
    q   = quant[row].numpy()

    axes[row, 0].imshow(t2,  cmap='gray', vmin=0, vmax=1)
    axes[row, 1].imshow(dwi, cmap='gray', vmin=0, vmax=1)

    # Mask (show -1 as blank)
    if seg.min() < 0:
        axes[row, 2].text(0.5, 0.5, 'Unlabeled', ha='center', va='center',
                          transform=axes[row, 2].transAxes, fontsize=11)
        axes[row, 2].set_facecolor('#222')
    else:
        axes[row, 2].imshow(seg, cmap=cmap_seg, vmin=0, vmax=1)

    # Overlay
    axes[row, 3].imshow(t2, cmap='gray', vmin=0, vmax=1)
    if seg.min() >= 0:
        alpha_mask = np.where(seg > 0, 0.45, 0)
        axes[row, 3].imshow(seg, cmap='Reds', alpha=alpha_mask, vmin=0, vmax=1)
        if q[0] >= 0:
            cx, cy = int(q[0]*256), int(q[1]*256)
            axes[row, 3].plot(cx, cy, 'b+', markersize=10, markeredgewidth=2)

    axes[row, 0].set_ylabel(ids[row], fontsize=8)
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Sample Batch: T2FS / DWI / Mask / Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_batch.png', dpi=100, bbox_inches='tight')
plt.show()
print('Quant (x_norm, y_norm, area_norm):', quant.numpy().round(3))

## 3. Model Architecture

In [ ]:
from models.meamt_net import MEaMtNet
from models.distillation import Distiller

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m_cfg  = cfg['model']

student = MEaMtNet(m_cfg).to(device)
teacher = MEaMtNet(m_cfg).to(device)

# Parameter counts
def count_params(model):
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(student)
print(f'MEaMtNet parameters: {total/1e6:.2f} M  (trainable: {trainable/1e6:.2f} M)')

In [ ]:
# Single forward pass
student.eval()
with torch.no_grad():
    dummy = torch.randn(2, 2, 256, 256).to(device)
    out   = student(dummy)

print('Output keys :', list(out.keys()))
print('seg logits  :', out['seg'].shape)          # [B, num_classes, H, W]
print('quant pred  :', out['quant'].shape)         # [B, 3]
print('features    :', out['features'].shape)      # [B, 1024, 16, 16]
if 'seg_alpha' in out:
    print('seg_alpha   :', out['seg_alpha'].shape)
    print('seg_uncert  :', out['seg_uncertainty'].shape)

In [ ]:
# Visualise architecture as module tree
def print_module_tree(module, prefix='', max_depth=2, depth=0):
    if depth > max_depth:
        return
    for name, child in module.named_children():
        params = sum(p.numel() for p in child.parameters()) / 1e3
        print(f'{prefix}{name}  [{child.__class__.__name__}]  {params:.1f}K params')
        print_module_tree(child, prefix + '  │  ', max_depth, depth + 1)

print('MEaMtNet')
print_module_tree(student, '  ', max_depth=2)

## 4. Loss Functions

In [ ]:
from losses.segmentation_loss import DiceCELoss
from losses.distillation_loss import DistillationLoss, ContrastiveLoss
from losses.evidence_loss import EvidenceLoss

l_cfg   = cfg['loss']
loss_seg     = DiceCELoss()
loss_distill = DistillationLoss(
    temperature=l_cfg['temperature'],
    feat_weight=l_cfg['distill_weight'],
    contra_weight=l_cfg['contrastive_weight'],
)
loss_evi = EvidenceLoss(beta=l_cfg['beta'])

# Quick smoke-test with random data
B, C, H, W = 2, 2, 256, 256
pred_seg  = torch.randn(B, C, H, W)
gt_seg    = torch.randint(0, C, (B, H, W))

l1 = loss_seg(pred_seg, gt_seg)
print(f'DiceCE Loss     : {l1.item():.4f}')

# Distillation loss (needs projections)
s_proj = torch.randn(B, 128)
t_proj = torch.randn(B, 128)
fake_out = {'student_proj': s_proj, 'teacher_proj': t_proj}
l2 = loss_distill(fake_out)
print(f'Distillation Loss: {l2.item():.4f}')

# Evidence loss (needs model output)
student.eval()
with torch.no_grad():
    out_s = student(torch.randn(B, 2, H, W).to(device))
l3 = loss_evi(out_s, gt_seg.to(device))
print(f'Evidence Loss   : {l3.item():.4f}')

In [ ]:
# Visualise contrastive loss landscape
temps = np.logspace(-2, 0, 50)
contra = ContrastiveLoss(temperature=0.07)

losses_same   = []   # similar projections
losses_random = []   # random projections

for t in temps:
    contra.temperature = float(t)
    s = torch.randn(8, 128); s = s / s.norm(dim=1, keepdim=True)
    noise = torch.randn_like(s) * 0.1
    losses_same.append(contra(s, s + noise).item())
    losses_random.append(contra(s, torch.randn(8, 128)).item())

plt.figure(figsize=(8, 4))
plt.semilogx(temps, losses_same,   label='Similar projections (positive pairs)')
plt.semilogx(temps, losses_random, label='Random projections')
plt.xlabel('Temperature τ'); plt.ylabel('InfoNCE Loss')
plt.title('Contrastive Loss vs Temperature'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('contrastive_loss.png', dpi=100); plt.show()

## 5. Training Loop (Mini Demo)

In [ ]:
import torch.optim as optim
from utils.scheduler import get_scheduler

distiller = Distiller(teacher, student).to(device)

optimizer = optim.AdamW(
    distiller.parameters(),
    lr=cfg['training']['learning_rate'],
    weight_decay=cfg['training']['weight_decay'],
)
scheduler = get_scheduler(optimizer, cfg)

# ── Run 5 demo epochs ─────────────────────────────────────────────────────────
DEMO_EPOCHS = 5
history = {'loss': [], 'seg': [], 'distill': [], 'evi': [], 'lr': []}

for epoch in range(DEMO_EPOCHS):
    distiller.train()
    ep_loss = ep_seg = ep_dist = ep_evi = 0.0

    for batch in train_loader:
        images  = batch['image'].to(device)
        masks   = batch['seg'].to(device)
        quant_gt = batch['quant'].to(device)
        labeled = batch['labeled'].bool().to(device)

        optimizer.zero_grad()
        out = distiller(images)

        # Supervised losses on labeled slice
        if labeled.any():
            seg_loss = loss_seg(out['seg'][labeled], masks[labeled].clamp(min=0))
            evi_loss = loss_evi(
                {k: v[labeled] if isinstance(v, torch.Tensor) and v.shape[0] == images.shape[0] else v
                 for k, v in out.items()},
                masks[labeled].clamp(min=0),
                quant_gt[labeled],
            )
        else:
            seg_loss = evi_loss = torch.tensor(0.0, device=device)

        distill_loss = loss_distill(out)

        total = (l_cfg['seg_weight']      * seg_loss
               + l_cfg['distill_weight']  * distill_loss
               + l_cfg['evidence_weight'] * evi_loss)

        total.backward()
        torch.nn.utils.clip_grad_norm_(distiller.parameters(), 5.0)
        optimizer.step()

        ep_loss  += total.item()
        ep_seg   += seg_loss.item()
        ep_dist  += distill_loss.item()
        ep_evi   += evi_loss.item()

    scheduler.step()
    n = len(train_loader)
    history['loss'].append(ep_loss / n)
    history['seg'].append(ep_seg / n)
    history['distill'].append(ep_dist / n)
    history['evi'].append(ep_evi / n)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    print(f'Epoch {epoch+1}/{DEMO_EPOCHS} | '
          f'Total={ep_loss/n:.4f}  Seg={ep_seg/n:.4f}  '
          f'Distill={ep_dist/n:.4f}  Evi={ep_evi/n:.4f}  '
          f'LR={optimizer.param_groups[0]["lr"]:.2e}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(history['loss'],   'k-o', label='Total')
axes[0].plot(history['seg'],    'b-o', label='Segmentation')
axes[0].plot(history['distill'],'g-o', label='Distillation')
axes[0].plot(history['evi'],    'r-o', label='Evidence')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['lr'], 'm-o')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')
axes[1].set_title('LR Schedule (Cosine Warmup)'); axes[1].grid(alpha=0.3)

loss_parts = np.array([history['seg'], history['distill'], history['evi']])
loss_parts = loss_parts / loss_parts.sum(axis=0, keepdims=True)
axes[2].stackplot(range(DEMO_EPOCHS), loss_parts,
                  labels=['Seg', 'Distill', 'Evidence'],
                  colors=['#4C72B0', '#55A868', '#C44E52'], alpha=0.8)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Relative contribution')
axes[2].set_title('Loss Composition'); axes[2].legend(loc='upper right'); axes[2].grid(alpha=0.3)

plt.suptitle('Training Summary', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Inference & Evaluation

In [ ]:
from eval.inference import run_inference, evaluate_model
from utils.metrics import compute_all_metrics

student.eval()
all_metrics = []

with torch.no_grad():
    for batch in val_loader:
        images   = batch['image'].to(device)
        gt_seg   = batch['seg'].to(device)
        gt_quant = batch['quant'].to(device)

        pred_seg, pred_quant, out = run_inference(student, images, tta=False)

        if (gt_seg >= 0).any():
            m = compute_all_metrics(pred_seg, gt_seg, pred_quant, gt_quant,
                                    cfg['dataset']['num_classes'])
            all_metrics.append(m)

mean = {k: float(np.nanmean([m[k] for m in all_metrics])) for k in all_metrics[0]}
print('Validation Metrics (demo — synthetic data, few epochs):')
for k, v in mean.items():
    unit = '' if k == 'dice' else (' px' if 'hd' in k or 'asd' in k else '')
    print(f'  {k:15s}: {v:.4f}{unit}')

In [ ]:
# Metrics bar chart
labels = list(mean.keys())
values = [mean[k] for k in labels]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

plt.figure(figsize=(8, 4))
bars = plt.bar(labels, values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.title('Validation Metrics'); plt.ylabel('Score')
plt.ylim(0, max(values) * 1.2)
plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig('val_metrics.png', dpi=100); plt.show()

## 7. Results Visualization

In [ ]:
import torch.nn.functional as F

student.eval()
batch = next(iter(val_loader))
images   = batch['image'].to(device)
gt_seg   = batch['seg'][0].numpy()
gt_quant = batch['quant'][0].numpy()

with torch.no_grad():
    pred_seg, pred_quant, out = run_inference(student, images, tta=False)

pred_cls  = pred_seg[0].argmax(dim=0).cpu().numpy()       # [H, W]
pred_prob = F.softmax(pred_seg[0], dim=0)[1].cpu().numpy()  # tumour probability

# Uncertainty
if 'seg_uncertainty' in out:
    uncert = out['seg_uncertainty'][0, 0].cpu().numpy()
else:
    uncert = None

t2  = images[0, 0].cpu().numpy()
dwi = images[0, 1].cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 0: inputs & ground truth
axes[0,0].imshow(t2, cmap='gray'); axes[0,0].set_title('T2FS')
axes[0,1].imshow(dwi, cmap='gray'); axes[0,1].set_title('DWI')
axes[0,2].imshow(gt_seg, cmap='Reds', vmin=0, vmax=1); axes[0,2].set_title('Ground Truth')
overlay_gt = np.stack([t2]*3, axis=-1)
overlay_gt[gt_seg==1, 0] = 0.9; overlay_gt[gt_seg==1, 1] = 0.2; overlay_gt[gt_seg==1, 2] = 0.2
axes[0,3].imshow(overlay_gt.clip(0,1)); axes[0,3].set_title('GT Overlay')
if gt_quant[0] >= 0:
    cx, cy = int(gt_quant[0]*256), int(gt_quant[1]*256)
    axes[0,3].plot(cx, cy, 'g+', markersize=12, markeredgewidth=2)

# Row 1: predictions
axes[1,0].imshow(pred_prob, cmap='hot', vmin=0, vmax=1); axes[1,0].set_title('Tumour Probability')
im = axes[1,1].imshow(pred_cls, cmap='Reds', vmin=0, vmax=1); axes[1,1].set_title('Predicted Mask')
overlay_pred = np.stack([t2]*3, axis=-1)
overlay_pred[pred_cls==1, 0] = 0.2; overlay_pred[pred_cls==1, 1] = 0.6; overlay_pred[pred_cls==1, 2] = 0.9
axes[1,2].imshow(overlay_pred.clip(0,1)); axes[1,2].set_title('Prediction Overlay')
px, py = int(pred_quant[0,0].item()*256), int(pred_quant[0,1].item()*256)
axes[1,2].plot(px, py, 'y+', markersize=12, markeredgewidth=2)

if uncert is not None:
    im_u = axes[1,3].imshow(uncert, cmap='plasma')
    plt.colorbar(im_u, ax=axes[1,3], fraction=0.046)
    axes[1,3].set_title('Epistemic Uncertainty')
else:
    axes[1,3].axis('off')

for ax in axes.flat:
    ax.axis('off')

red_p  = mpatches.Patch(color='#e33', label='Ground Truth')
blue_p = mpatches.Patch(color='#39f', label='Prediction')
fig.legend(handles=[red_p, blue_p], loc='lower center', ncol=2, fontsize=10)

plt.suptitle('Segmentation & Uncertainty Prediction', fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('prediction_results.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'GT  quant (x, y, area): {gt_quant.round(3)}')
print(f'Pred quant (x, y, area): {pred_quant[0].cpu().numpy().round(3)}')

In [ ]:
# Semi-supervised comparison: labeled fraction vs Dice
ratios = [0.05, 0.10, 0.20, 0.50, 1.00]
# Placeholder values — replace with actual experiment results
dice_ours    = [0.612, 0.683, 0.741, 0.788, 0.821]
dice_baseline= [0.543, 0.601, 0.658, 0.741, 0.815]
dice_full_sup= [0.821] * len(ratios)

plt.figure(figsize=(8, 5))
plt.plot([r*100 for r in ratios], dice_ours,     'b-o', label='MEaMt-Net (Ours)', linewidth=2)
plt.plot([r*100 for r in ratios], dice_baseline, 'r--s', label='Baseline SSL', linewidth=2)
plt.plot([r*100 for r in ratios], dice_full_sup, 'g:', label='Fully supervised (100%)', linewidth=1.5)
plt.xlabel('Label Ratio (%)', fontsize=12)
plt.ylabel('Dice Score', fontsize=12)
plt.title('Semi-Supervised Performance on LLD-MMRI', fontsize=13)
plt.legend(fontsize=10); plt.grid(alpha=0.3)
plt.xticks([r*100 for r in ratios])
plt.tight_layout()
plt.savefig('ssl_comparison.png', dpi=100)
plt.show()

## Summary

| Component | Description |
|-----------|-------------|
| **Backbone** | Edge-Guided Attention + Mamba SSM bottleneck |
| **Decoders** | UNet-style segmentation + regression quantification |
| **Uncertainty** | Dirichlet evidence head (epistemic uncertainty) |
| **Distillation** | Teacher (CE-MRI) → Student (T2FS+DWI) via InfoNCE |
| **Semi-supervised** | 10–20% labeled data + uncertainty-guided consistency |
| **Metrics** | Dice / HD95 / ASD (seg) + MAE X,Y,Area (quant) |

To run the full training pipeline:
```bash
python train.py --config configs/default.yaml
```

To evaluate:
```bash
python test.py --config configs/default.yaml --checkpoint checkpoints/meamtnet_best.pth --tta
```